# Task 1 — Build & Evaluate a Linear Regression Model
## House Price Predictor — California Housing Dataset
**Maincrafts Technology | AI & ML Internship**

---

For this task I picked the California Housing dataset — it felt like a practical choice since house price prediction is something you actually see in the real world. The goal is to build a linear regression model and see how well it can predict median house values based on features like income, location, rooms, etc.

Let me walk through what I did step by step.

## Step 1 — Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

plt.rcParams['figure.dpi'] = 120
sns.set_theme(style='whitegrid')

## Step 2 — Load Dataset

The dataset comes built into scikit-learn which is convenient. It has 8 features and ~20k rows. The target column `MedHouseVal` is median house value in units of $100k.

In [ ]:
data = fetch_california_housing(as_frame=True)
df = pd.concat([data.data, data.target.rename('MedHouseVal')], axis=1)

print('Shape:', df.shape)
df.head()

In [ ]:
# quick look at data types and if anything looks off
df.info()

## Step 3 — Exploratory Data Analysis

Before jumping into modelling I wanted to understand the data a bit. First checking for missing values, then looking at distributions.

In [ ]:
# no missing values which is nice, saves preprocessing time
print('Missing values:')
print(df.isnull().sum())

In [ ]:
df.describe().round(2)

In [ ]:
# AveRooms and AveOccup have some crazy high max values — probably outliers worth noting
# AveRooms max is 141 which doesn't make sense for an average, likely a data issue
print('AveRooms outliers (>20):', (df['AveRooms'] > 20).sum())
print('AveOccup outliers (>10):', (df['AveOccup'] > 10).sum())

In [ ]:
# plotting feature distributions to see what we're dealing with
features = data.feature_names
fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for i, feat in enumerate(features):
    axes[i//4][i%4].hist(df[feat], bins=40, color='steelblue', edgecolor='white', alpha=0.85)
    axes[i//4][i%4].set_title(feat, fontsize=10, fontweight='bold')

plt.suptitle('Feature Distributions — California Housing', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# MedInc looks roughly normal-ish, most others are right-skewed

In [ ]:
# target distribution
plt.figure(figsize=(7, 4))
df['MedHouseVal'].hist(bins=50, color='coral', edgecolor='white')
plt.title('Target: Median House Value', fontweight='bold')
plt.xlabel('Median House Value ($100k)')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

# there's a clear spike at 5.0 — that's likely because values were capped at $500k in the original data

In [ ]:
# correlation heatmap
plt.figure(figsize=(10, 7))
sns.heatmap(df.corr(numeric_only=True), annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.5)
plt.title('Correlation Matrix', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
corr_target = df.corr(numeric_only=True)['MedHouseVal'].sort_values(ascending=False).round(3)
print('Correlation with target:')
print(corr_target)

**Observation:** `MedInc` (median income) has by far the strongest correlation with house value at ~0.688. Makes sense — wealthier areas = pricier homes. `Latitude` is negatively correlated (~-0.14) which is interesting, probably reflecting that northern California tends to have lower prices than the Bay Area / LA.

`AveBedrms` and `AveRooms` are highly correlated with each other (multicollinearity), which could be a minor issue for linear regression but I'll keep both for now.

## Step 4 — Train/Test Split

Using all 8 features. 80/20 split with random_state=42 so results are reproducible.

In [ ]:
X = df.drop(columns='MedHouseVal')
y = df['MedHouseVal']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Train: {X_train.shape}, Test: {X_test.shape}')

## Step 5 — Model Training

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print(f'Intercept: {model.intercept_:.4f}')

In [ ]:
# checking which features the model weighted most heavily
coef_df = pd.DataFrame({'Feature': X.columns, 'Coefficient': model.coef_})
coef_df = coef_df.sort_values('Coefficient', key=abs, ascending=False)
print(coef_df.to_string(index=False))

# interesting that Latitude has a larger coefficient than MedInc even though MedInc
# had stronger correlation — features aren't scaled so this comparison is a bit misleading

## Step 6 — Evaluation

In [ ]:
mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2   = r2_score(y_test, y_pred)

print(f'MAE  : {mae:.4f}')
print(f'RMSE : {rmse:.4f}')
print(f'R2   : {r2:.4f}')
print(f'\nSo the model explains {r2*100:.1f}% of the variance in prices.')
print(f'On average predictions are off by ~${mae*100:.0f}k — not bad but not great either.')

**My take on the results:**

R2 of ~0.60 means the model is doing okay but missing a lot. Linear regression is probably hitting its ceiling here — the relationship between location/income and house prices likely isn't purely linear. The ~$52k average error is decent for a baseline but I think a tree-based model would do noticeably better.

Also worth noting — the capped values at $500k probably hurt performance too since the model can't learn what drives the highest-priced homes.

## Step 7 — Visualizations

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# actual vs predicted
axes[0].scatter(y_test, y_pred, alpha=0.3, color='steelblue', s=8)
lims = [min(y_test.min(), y_pred.min())-0.1, max(y_test.max(), y_pred.max())+0.1]
axes[0].plot(lims, lims, 'r--', linewidth=1.8, label='Perfect Fit')
axes[0].set_xlabel('Actual ($100k)')
axes[0].set_ylabel('Predicted ($100k)')
axes[0].set_title('Actual vs Predicted', fontweight='bold')
axes[0].legend()

# residuals
residuals = y_test - y_pred
axes[1].scatter(y_pred, residuals, alpha=0.3, color='darkorange', s=8)
axes[1].axhline(0, color='red', linestyle='--', linewidth=1.8)
axes[1].set_xlabel('Predicted ($100k)')
axes[1].set_ylabel('Residual')
axes[1].set_title('Residual Plot', fontweight='bold')

plt.tight_layout()
plt.show()

# the flat band of predictions around 5.0 on the right is from the capped values in the dataset

In [ ]:
# coefficient plot
colors = ['steelblue' if c > 0 else 'tomato' for c in coef_df['Coefficient']]
plt.figure(figsize=(8, 5))
plt.barh(coef_df['Feature'], coef_df['Coefficient'], color=colors)
plt.axvline(0, color='black', linewidth=0.8)
plt.title('Feature Coefficients', fontweight='bold')
plt.xlabel('Coefficient Value')
plt.tight_layout()
plt.show()

## Step 8 — Save Model

In [ ]:
joblib.dump(model, 'linear_regression_model.pkl')

# reload and test on a few samples
loaded_model = joblib.load('linear_regression_model.pkl')
sample = X_test.iloc[:3]
preds = loaded_model.predict(sample)

print('Sanity check — sample predictions:')
for i, p in enumerate(preds):
    print(f'  Sample {i+1}: predicted ${p*100:.1f}k | actual ${y_test.iloc[i]*100:.1f}k')

## What I'd Try Next

The linear regression gives a reasonable baseline (~60% R2) but there's clearly room to improve:

- **Feature scaling** — would make coefficients more interpretable and is needed before trying Ridge/Lasso
- **Remove/cap outliers** — the extreme AveRooms/AveOccup values are probably hurting the model
- **Polynomial features** — income vs price relationship is likely non-linear
- **Random Forest or XGBoost** — tree models handle non-linearity naturally and would probably push R2 to 0.80+
- **Cross-validation** — 80/20 single split can be a bit lucky/unlucky, k-fold would give a more honest estimate

---
*Task 1 done — Maincrafts Technology AI & ML Internship*